# Test Your Algorithm

## Instructions
1. From the **Pulse Rate Algorithm** Notebook you can do one of the following:
   - Copy over all the **Code** section to the following Code block.
   - Download as a Python (`.py`) and copy the code to the following Code block.
2. In the bottom right, click the <span style="color:blue">Test Run</span> button. 

### Didn't Pass
If your code didn't pass the test, go back to the previous Concept or to your local setup and continue iterating on your algorithm and try to bring your training error down before testing again.

### Pass
If your code passes the test, complete the following! You **must** include a screenshot of your code and the Test being **Passed**. Here is what the starter filler code looks like when the test is run and should be similar. A passed test will include in the notebook a green outline plus a box with **Test passed:** and in the Results bar at the bottom the progress bar will be at 100% plus a checkmark with **All cells passed**.
![Example](example.png)

1. Take a screenshot of your code passing the test, make sure it is in the format `.png`. If not a `.png` image, you will have to edit the Markdown render the image after Step 3. Here is an example of what the `passed.png` would look like 
2. Upload the screenshot to the same folder or directory as this jupyter notebook.
3. Rename the screenshot to `passed.png` and it should show up below.
![Passed](passed.png)
4. Download this jupyter notebook as a `.pdf` file. 
5. Continue to Part 2 of the Project. 

In [ ]:
import glob

import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import scipy.io as spio
from scipy.signal import butter, filtfilt


fs = 125

def LoadTroikaDataset():
    """
    Retrieve the .mat filenames for the troika dataset.

    Review the README in ./datasets/troika/ to understand the organization of the .mat files.

    Returns:
        data_fls: Names of the .mat files that contain signal data
        ref_fls: Names of the .mat files that contain reference data
        <data_fls> and <ref_fls> are ordered correspondingly, so that ref_fls[5] is the
            reference data for data_fls[5], etc...
    """
    data_dir = "./datasets/troika/training_data"
    data_fls = sorted(glob.glob(data_dir + "/DATA_*.mat"))
    ref_fls = sorted(glob.glob(data_dir + "/REF_*.mat"))
    return data_fls, ref_fls

def LoadTroikaDataFile(data_fl):
    """
    Loads and extracts signals from a troika data file.

    Usage:
        data_fls, ref_fls = LoadTroikaDataset()
        ppg, accx, accy, accz = LoadTroikaDataFile(data_fls[0])

    Args:
        data_fl: (str) filepath to a troika .mat file.

    Returns:
        numpy arrays for ppg, accx, accy, accz signals.
    """
    data = spio.loadmat(data_fl)["sig"]
    return data[2:]

def _bandpass_filter(signal):
    """Bandpass filter the signal between 40 and 240 BPM."""
    b, a = butter(3, (40/60.0, 240/60.0), btype='bandpass', fs=fs)
    return filtfilt(b, a, signal)


def RunPulseRateAlgorithm(data_fl, ref_fl):
    """
    Estimate per-frame pulse-rate error and confidence for one Troika trial.
    
    Args:
        data_fl: Path to one Troika signal file (DATA_*.mat).
        ref_fl: Path to the matching Troika reference BPM file (REF_*.mat).

    Returns:
        errors: NumPy array of absolute BPM errors per frame.
        confidence: NumPy array of confidence scores per frame.
    """
    # Load trial signals and reference BPM track.
    ppg, accx, accy, accz = LoadTroikaDataFile(data_fl)
    ref_bpm = spio.loadmat(ref_fl)["BPM0"].squeeze()

    # Compute accelerometer magnitude channel.
    acc_mag = np.sqrt(np.sum(np.square(np.vstack((accx, accy, accz))), axis=0))

    # Bandpass both PPG and motion signals in the physiological HR band.
    filt_ppg = _bandpass_filter(ppg)
    filt_accmag = _bandpass_filter(acc_mag)

    # The Troika reference is aligned to 8-second windows with 6-second overlap.
    ppg_spec, ppg_freqs, _, _ = plt.specgram(filt_ppg, Fs=fs, NFFT=8 * fs, noverlap=6 * fs)
    accmag_spec, accmag_freqs, _, _ = plt.specgram(filt_accmag, Fs=fs, NFFT=8 * fs, noverlap=6 * fs)

    n_frames = min(accmag_spec.shape[1], ppg_spec.shape[1], ref_bpm.shape[0])
    if n_frames == 0:
        return np.array([]), np.array([])

    errors = np.empty(n_frames, dtype=float)
    confidence = np.empty(n_frames, dtype=float)
    est_bpm = 0.0
    band_half_width_hz = 40.0 / 60.0

    for frame_idx in range(n_frames):
        acc_slice = accmag_spec[:, frame_idx]
        ppg_slice = ppg_spec[:, frame_idx]

        # Start with dominant spectral peaks from motion and PPG.
        acc_peak_idx = int(np.argmax(acc_slice))
        ppg_peak_idx = int(np.argmax(ppg_slice))

        if acc_peak_idx != ppg_peak_idx:
            # If peaks differ, trust the dominant PPG candidate.
            est_bpm = float(ppg_freqs[ppg_peak_idx] * 60.0)
        else:
            # If peaks collide, probe additional strong candidates with a tiny boolean mask.
            k = min(4, acc_slice.size)
            acc_top_idx = np.argpartition(acc_slice, -k)[-k:]
            ppg_top_idx = np.argpartition(ppg_slice, -k)[-k:]
            ppg_top_idx = ppg_top_idx[np.argsort(ppg_slice[ppg_top_idx])[::-1]]

            acc_mask = np.zeros(acc_slice.size, dtype=bool)
            acc_mask[acc_top_idx] = True

            est_bpm = float(ppg_freqs[ppg_peak_idx] * 60.0)
            for p_idx in ppg_top_idx:
                if not acc_mask[int(p_idx)]:
                    est_bpm = float(ppg_freqs[int(p_idx)] * 60.0)
                    break

        true_bpm = float(ref_bpm[frame_idx])
        errors[frame_idx] = np.abs(true_bpm - est_bpm)

        near_estimate = (ppg_freqs > (est_bpm / 60.0) - band_half_width_hz) & (ppg_freqs < (est_bpm / 60.0) + band_half_width_hz)

        # Confidence is spectral concentration around the estimated frequency.
        ppg_power_local = float(np.sum(ppg_slice[near_estimate]))
        ppg_power_total = float(np.sum(ppg_slice)) + 1e-12
        confidence[frame_idx] = ppg_power_local / ppg_power_total

    return errors, confidence